# Stage E only – Google Colab

Run **Stage E** (LLM extraction) in Colab with GPU. Use this when you ran stages a–d locally and want only the final step in the cloud.

**Setup:** Runtime → Change runtime type → **T4 GPU**. Then run cells in order.

**You need:**
1. `Thesis_llama_colab.zip` (from `create_colab_zip.py`)
2. `stage_d_candidate_statements.json` (from `outputs/STAGE_D_v1/` after running Stage D locally)

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers accelerate torch sentencepiece
!pip install -q nltk pydantic huggingface_hub
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("[OK] Dependencies installed!")

## Step 2: Login to Hugging Face (required for Llama)

Get a token from https://huggingface.co/settings/tokens and accept the Llama license: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import getpass
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("[OK] Logged in (Colab Secrets)")
except:
    hf_token = getpass.getpass("Hugging Face token: ")
    login(token=hf_token)
    print("[OK] Logged in")

## Step 3: Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable T4 GPU: Runtime → Change runtime type"
print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")

## Step 4: Upload and extract ZIP (pipeline code)

In [ ]:
from google.colab import files
import zipfile
import os
import sys

print("[*] Upload Thesis_llama_colab.zip...")
uploaded = files.upload()
for fn in uploaded:
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn, 'r') as z:
            z.extractall('/content')
        break
content_list = os.listdir('/content')
if 'pipeline' in content_list and os.path.isdir('/content/pipeline'):
    project_dir = '/content'
else:
    project_dir = '/content'
    for d in content_list:
        if os.path.isdir(f'/content/{d}') and os.path.isdir(f'/content/{d}/pipeline'):
            project_dir = f'/content/{d}'
            break
os.chdir(project_dir)
sys.path.insert(0, project_dir)
print(f"[OK] Project: {project_dir}")

## Step 5: Upload Stage D output

In [ ]:
from google.colab import files
import os

STAGE_D_PATH = "/content/stage_d_candidate_statements.json"
print("[*] Upload stage_d_candidate_statements.json (from outputs/STAGE_D_v1/ on your PC)...")
uploaded = files.upload()
if not uploaded:
    print("[!] No file uploaded. Run this cell again and select the JSON file.")
else:
    fn = list(uploaded.keys())[0]
    # In Colab, uploaded[fn] is the file content (bytes)
    data = uploaded[fn]
    if not isinstance(data, bytes):
        data = open(fn, 'rb').read()
    with open(STAGE_D_PATH, 'wb') as out:
        out.write(data)
    size_mb = len(data) / (1024 * 1024)
    print(f"[OK] Saved to {STAGE_D_PATH} ({size_mb:.2f} MB)")
    if size_mb < 0.01:
        print("[!] File is very small — make sure you selected the full stage_d_candidate_statements.json")

## Step 6: Run Stage E

Loads Stage D file, runs LLM on statements (extract triples) and on table_triples (validate/split), saves `stage_e_validated_output.json`.

In [ ]:
import sys
import os
import json
import time

STAGE_D_PATH = "/content/stage_d_candidate_statements.json"
STAGE_E_OUTPUT = "stage_e_validated_output.json"
VALIDATION_MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
BATCH_SIZE = 4
MAX_EXTRACTION_TOKENS = 400

if not os.path.exists(STAGE_D_PATH):
    raise FileNotFoundError(f"Upload Stage D file first (Step 5). Expected: {STAGE_D_PATH}")

size = os.path.getsize(STAGE_D_PATH)
if size == 0:
    raise ValueError(f"{STAGE_D_PATH} is empty. Re-run Step 5 and upload stage_d_candidate_statements.json again.")

content_list = os.listdir('/content')
project_path = '/content' if 'pipeline' in content_list and os.path.isdir('/content/pipeline') else None
if not project_path:
    for d in content_list:
        if os.path.isdir(f'/content/{d}') and os.path.isdir(f'/content/{d}/pipeline'):
            project_path = f'/content/{d}'
            break
project_path = project_path or '/content'
sys.path.insert(0, project_path)
os.chdir(project_path)

from pipeline.data import CandidateStatements, ValidatedFactsAndQualifiers
from pipeline.models import ValidationModel
from pipeline.inference import Validate

try:
    with open(STAGE_D_PATH, 'r', encoding='utf-8') as f:
        stage_d_data = json.load(f)
except json.JSONDecodeError as e:
    raise ValueError(
        f"Invalid JSON in {STAGE_D_PATH} (file may be empty or corrupted). "
        "Re-run Step 5 and upload the correct stage_d_candidate_statements.json from outputs/STAGE_D_v1/"
    ) from e
if "statements" not in stage_d_data:
    raise KeyError(f"File must contain 'statements' key (Stage D output). Got: {list(stage_d_data.keys())[:5]}")

candidate_statements = CandidateStatements()
for stmt in stage_d_data["statements"]:
    candidate_statements.add_statement(stmt)
table_triples = stage_d_data.get("table_triples", [])
print(f"[*] Loaded: {candidate_statements.count()} statements, {len(table_triples)} table triples")

print("[*] Loading LLM...")
validation_model = ValidationModel(model_name=VALIDATION_MODEL_NAME)
validator = Validate(validation_model=validation_model, batch_size=BATCH_SIZE, max_new_tokens=MAX_EXTRACTION_TOKENS)

start = time.time()
print("[*] Stage E: Extracting triples from text...")
validated_facts = validator.validate(candidate_statements)
if table_triples:
    print("[*] Stage E: Table triples through LLM...")
    validator.validate_table_triples(validated_facts, table_triples)
elapsed = time.time() - start

output_data = {
    "metadata": {"stage": "e", "total_statements": validated_facts.count(), "extraction_model": VALIDATION_MODEL_NAME, "table_triples_through_llm": len(table_triples)},
    "validated_statements": validated_facts.get_all(),
}
with open(STAGE_E_OUTPUT, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"\n[SUCCESS] Stage E complete in {elapsed/60:.1f} min. Output: {STAGE_E_OUTPUT}")

## Step 7: Download result

In [ ]:
from google.colab import files
import os

if os.path.exists("stage_e_validated_output.json"):
    files.download("stage_e_validated_output.json")
    print("[OK] Downloaded")
else:
    print("[!] File not found. Run Step 6 first.")